### 1. Secure Initialization, Pydantic Schema, and Configuration

This block establishes the entire environment, securely loads the API key, and defines critical pipeline variables like the VIDEO_URL and WHISPER_MODEL. Its most important function is defining the Pydantic schema, which forces the Gemini model to output structured JSON data, including the requirement for the original asset name and the removal of the unnecessary confidence score.

In [ ]:
# --- Block 1: Secure Initialization and Configuration Loader ---
import os
import re
import subprocess
import whisper
import warnings
import json 
import yaml
from dotenv import load_dotenv

from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError 
from typing import Literal, List
from datetime import datetime
import time 

# Environment Setup and Configuration Loading
load_dotenv()
warnings.filterwarnings("ignore") 

# Load configuration from YAML file
try:
    with open('config.yaml', 'r') as f:
        CONFIG = yaml.safe_load(f)
except FileNotFoundError:
    raise FileNotFoundError("Error: 'config.yaml' file not found. Please create it.")
except yaml.YAMLError as e:
    raise ValueError(f"Error parsing 'config.yaml': {e}")


# Map YAML config to global variables
VIDEO_LIST_PATH = CONFIG['INPUTS']['VIDEO_LIST_PATH'] # NEW VARIABLE
MAX_LLM_RETRIES = CONFIG['INPUTS']['MAX_LLM_RETRIES']
WHISPER_MODEL = CONFIG['MODEL_PERFORMANCE']['WHISPER_MODEL']
GEMINI_MODEL = CONFIG['MODEL_PERFORMANCE']['GEMINI_MODEL']
CACHE_DIR = CONFIG['DIRECTORIES']['CACHE_DIR']
SUMMARY_DIR = CONFIG['DIRECTORIES']['SUMMARY_DIR']

# Function to load video URLs from the list file
def load_video_urls(file_path: str) -> List[str]:
    """Reads and returns a list of valid URLs from a text file."""
    urls = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                url = line.strip()
                if url and not url.startswith('#'):
                    urls.append(url)
        if not urls:
             raise ValueError(f"The file '{file_path}' contains no valid video URLs.")
        return urls
    except FileNotFoundError:
        raise FileNotFoundError(f"Error: Video list file not found at '{file_path}'.")

# --- FINAL Pydantic Schema  ---
class TradeIdea(BaseModel):
    """A single stock trade idea extracted from the video."""
    ticker: str = Field(..., description="The primary NASDAQ stock ticker discussed. MUST be a valid ticker.")
    segment_timestamp: str = Field(..., description="The video timestamp (HH:MM:SS) where this trade idea starts.")
    key_thesis_tr: str = Field(..., description="A single, CONCISE summary, including the ASSET NAME/TICKER, combining all commentary (fundamental and technical) IN TURKISH.")
    sentiment: Literal['BULLISH', 'BEARISH', 'NEUTRAL'] = Field(..., description="The overall sentiment.")
    risk_assessment: Literal['High', 'Medium', 'Low'] = Field(..., description="Analyst's assessment of the risk/volatility of this trade.")
    risk_justification_tr: str = Field(..., description="A MINIMALIST SUMMARY of the exact phrase or reason the analyst gives for the risk assessment IN TURKISH. MUST be an empty string ('') if no specific reason is mentioned.")
    investment_horizon: Literal['Short-Term', 'Long-Term', 'Swing/Momentum'] = Field(..., description="The intended holding period.")
    trade_action: Literal['Long', 'Short', 'Hold', 'Watchlist'] = Field(..., description="The recommended action.")
    support_level: float = Field(..., description="The technical support price. Use 0.0 if not mentioned.")
    resistance_level: float = Field(..., description="The technical resistance price. Use 0.0 if not mentioned.")
    target_price: float = Field(..., description="The explicit long-term price target. Use 0.0 if not mentioned.")

class TradeSummaryList(BaseModel):
    """A list container for all trade ideas extracted from the video."""
    trade_ideas: List[TradeIdea] = Field(..., description="A list of all distinct stock trade ideas discussed in the video.")
    
# Initial setup prints
print(f"✅ Setup complete. Target: Video list file at {VIDEO_LIST_PATH}")
print(f"   Using Models: Whisper ({WHISPER_MODEL}), Gemini ({GEMINI_MODEL})")

### Module 1: Get Audio File (Caching)

This is the data acquisition module responsible for efficiently obtaining the video's audio track. It first checks a local cache directory for the existing .mp3 file based on the video ID, preventing redundant downloads on repeat runs. If not found, it uses the yt-dlp library to download and extract the audio stream.

In [ ]:
# --- Block 2: Module 1: Get Audio File (Download/Cache) ---

def module_get_audio_file(url: str) -> str:
    """Downloads and caches the audio file from a YouTube URL."""
    video_id_match = re.search(r'(?<=v=)[\w-]+', url)
    if not video_id_match:
        raise ValueError(f"Invalid YouTube URL: {url}")
    video_id = video_id_match.group(0)
    
    # 1. Ensure Cache Directory Exists
    os.makedirs(CACHE_DIR, exist_ok=True)
    
    # 2. Define File Paths
    audio_file_path = os.path.join(CACHE_DIR, f"{video_id}_audio.mp3")

    # 3. Check Cache
    if os.path.exists(audio_file_path):
        print(f"  1/3: Audio found in cache: {audio_file_path}")
        return audio_file_path

    # 4. Download Audio using yt-dlp
    print(f"  1/3: Audio not found. Downloading and converting to MP3...")
    try:
        subprocess.run(
            [
                "yt-dlp",
                "-x",  # Extract audio
                "--audio-format", "mp3", 
                "--audio-quality", "0", 
                "-o", audio_file_path, 
                url
            ],
            check=True,
            capture_output=True,
            text=True
        )
        print("✅ Audio download complete.")
        return audio_file_path
    except subprocess.CalledProcessError as e:
        print(f"❌ Error during audio download (yt-dlp error): {e.stderr}")
        raise
    except FileNotFoundError:
        print("❌ Error: 'yt-dlp' command not found. Please ensure it's installed and in your PATH.")
        raise

### Module 2: Create Transcription (Caching)

This block converts the raw audio into the Turkish text necessary for AI analysis. It first checks the local cache for an existing transcript to maximize efficiency. If uncached, it uses the specified medium Whisper model to perform the transcription, which is then saved to the cache for future use.

In [ ]:
# --- Block 3: Module 2: Create Transcription (Caching) ---

def module_create_transcription(audio_path: str) -> str:
    """Creates a Turkish transcription of the audio file."""
    video_id = os.path.basename(audio_path).split('_')[0]
    transcript_file_path = os.path.join(CACHE_DIR, f"{video_id}_audio_tr_transcript.txt")
    
    # 1. Check Cache
    if os.path.exists(transcript_file_path):
        print(f"  2/3: Transcript found in cache: {transcript_file_path}")
        with open(transcript_file_path, 'r', encoding='utf-8') as f:
            return f.read()

    # 2. Transcribe Audio
    print(f"  2/3: Transcript not found. Loading Whisper '{WHISPER_MODEL}' and transcribing...")
    
    try:
        model = whisper.load_model(WHISPER_MODEL)
    except Exception as e:
        print(f"❌ Error loading Whisper model: {e}")
        print("💡 NOTE: The 'medium' model is computationally intensive on CPU. Consider using 'small' if this step is too slow.")
        raise
        
    result = model.transcribe(audio_path, language="tr", task="transcribe")
    transcript_text = result["text"].strip()
    
    # 3. Save to Cache
    with open(transcript_file_path, 'w', encoding='utf-8') as f:
        f.write(transcript_text)
        
    print("✅ Transcription complete and cached.")
    return transcript_text

### Module 3: AI Analysis, Report Generation, and Execution

This final block is the intelligence engine and reporting stage. It uses the Gemini model with a strict system instruction to analyze the Turkish transcript and convert it into the required structured trade ideas (JSON). Finally, it runs the full pipeline and uses the structured data to generate and save the clean, streamlined executive summary report, omitting the zero-confidence field.

In [ ]:
# --- Block 4: Module 3: AI Analysis, Report Generation, and Execution  ---

def module_prepare_trade_summary(transcript_text: str) -> TradeSummaryList:
    # ... (function content remains unchanged, using the new instructions) ...
    """
    Uses the Gemini API to analyze the transcript and extract a structured list 
    with retry logic for robustness.
    """
    if not transcript_text.strip():
        raise ValueError("Cannot extract trade idea: Transcript text is empty.")
        
    client = genai.Client()
    
    # Base System Instruction: Incorporates CONCISE and ASSET NAME requirements
    system_instruction_base = (
        "You are an experienced, knowledgeable, and multilingual stock analyst and options trader. "
        "The input transcript is in TURKISH. Your primary goal is to **accurately and confidently identify the stock ticker** and **segment timestamp** for every distinct trade idea. "
        
        "**TICKER IDENTIFICATION PRIORITY:** 1. Contextually correct ticker. 2. Aggressive phonetic conversion. 3. Avoid 'N/A'."
        
        "**THESIS AND RISK REQUIREMENT (UPDATED):** **CONCISELY** summarize and combine all commentary (fundamental, technical, etc.) into the **key_thesis_tr** field. This summary **MUST** include the **asset name/ticker** for clarity. For the **risk_justification_tr** field, use a **MINIMALIST SUMMARY** of the reason. If the analyst does not give a specific reason, you MUST use an **empty string ('')** to indicate 'Not Specified', and **NOT** a translated phrase."
        
        "**FIDELITY REQUIREMENT:** All Turkish text fields MUST be near-verbatim extractions or summaries using ONLY the analyst's words from the provided transcript. **DO NOT introduce new vocabulary or concepts.**"
        "You must analyze the video transcript, extract ALL distinct trade ideas, and structure them into a list adhering to the provided JSON schema (TradeSummaryList)."
    )
    
    print("  3/3: Sending to AI for structured analysis...")
    
    final_trade_summary = None
    
    # LLM RETRY LOGIC for Robustness
    for attempt in range(MAX_LLM_RETRIES): 
        current_instruction = system_instruction_base
        if attempt > 0:
            print(f"⚠️ Retrying AI analysis... (Attempt {attempt + 1}/{MAX_LLM_RETRIES})")
            current_instruction += "\n\nCRITICAL: Your previous output failed JSON validation. You MUST ensure the output is a *perfectly* structured JSON list of TradeIdea objects. DO NOT include *any* text outside the JSON object delimiters."
            time.sleep(1) 

        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL, 
                contents=f"Analyze the following stock video transcript (Turkish) and extract ALL trade ideas, including the timestamp:\\n\\n---\\n{transcript_text}",
                config=types.GenerateContentConfig(
                    system_instruction=current_instruction,
                    response_mime_type="application/json",
                    response_schema=TradeSummaryList,
                ),
            )
            
            final_trade_summary = TradeSummaryList.model_validate(json.loads(response.text))
            print("✅ AI analysis successful.")
            return final_trade_summary
            
        except (ValidationError, json.JSONDecodeError) as e:
            if attempt == MAX_LLM_RETRIES - 1:
                 print("\n❌ CRITICAL ERROR: Pydantic Validation/JSON Decoding failed after all retries.")
                 raise ValueError(f"Final Validation error: {e}")
            pass
        except Exception as e:
            raise e

# --- HELPER FUNCTION for Reporting  ---
def format_price(value: float) -> str:
    """Returns a formatted price string ($X.XX) or a blank string if the value is 0.0."""
    return f"${value:.2f}" if value > 0.0 else ""

# --- Execution Pipeline (Main Block - NOW LOOPS OVER ALL URLs) ---

# Main function to process a single URL 
def process_single_video(video_url: str):
    print(f"\n🎬 Starting analysis for: {video_url}")

    try:
        # 1. Get the Audio File (Cached or Downloaded)
        audio_file_path = module_get_audio_file(video_url)
        
        # 2. Get the Transcript (Cached or Transcribed)
        transcript = module_create_transcription(audio_file_path)

        # 3. Prepare the Trade Summary (AI Analysis)
        text_id = re.search(r'(?<=v=)[\w-]+', video_url).group(0) # Use this ID for report

        trade_summary_list = module_prepare_trade_summary(transcript)
        
        print("\n" + "="*50)
        print(f"ALL MODULES COMPLETE for {text_id}. GENERATING REPORT.")
        print("="*50)

        # --- DYNAMIC FILENAME & DIRECTORY GENERATION ---
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        os.makedirs(SUMMARY_DIR, exist_ok=True)
        dynamic_report_filename = os.path.join(SUMMARY_DIR, f"summary_{text_id}_{timestamp}.txt")

        # Prepare the Executive Summary text file content
        report_content = f"--- EXECUTIVE TRADE SUMMARY REPORT ---\n"
        report_content += f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
        report_content += f"Source Video: {video_url}\n"
        report_content += f"Report File: {dynamic_report_filename}\n"
        report_content += f"-------------------------------------\n\n"
        
        # Report Generation Loop
        if not trade_summary_list.trade_ideas:
            report_content += "No distinct trade ideas were extracted from the video.\n"
        else:
            for i, trade_data in enumerate(trade_summary_list.trade_ideas):
                report_content += f"=== TRADE IDEA {i+1}: {trade_data.ticker} (Starts at {trade_data.segment_timestamp}) ===\n" 
                report_content += f"  Action:        {trade_data.trade_action}\n"
                report_content += f"  Sentiment:     {trade_data.sentiment}\n"
                report_content += f"  Horizon:       {trade_data.investment_horizon}\n" 
                report_content += f"  Risk:          {trade_data.risk_assessment}\n" 
                
                # CONDITIONAL PRINT: Only show Risk Reason if the field is not an empty string.
                if trade_data.risk_justification_tr:
                    report_content += f"  Risk Reason:   {trade_data.risk_justification_tr}\n" 
                    
                report_content += f"  Support:       {format_price(trade_data.support_level)}\n"
                report_content += f"  Resistance:    {format_price(trade_data.resistance_level)}\n"
                report_content += f"  Target Price:  {format_price(trade_data.target_price)}\n" 
                report_content += f"-------------------------------------\n"
                # Single Key Thesis
                report_content += f"  Key Thesis (TR Original):\n{trade_data.key_thesis_tr}\n" 
                report_content += "-------------------------------------\n\n"

        # Write the report to the text file
        with open(dynamic_report_filename, 'w', encoding='utf-8') as f:
            f.write(report_content)
            
        print(f"\n✅ Report successfully saved to {dynamic_report_filename}")
        
        # Print the full report content
        print("\n--- REPORT PREVIEW (Console) ---")
        print(report_content) 
        print("---------------------------------\n")


    except ValueError as ve:
        print(f"\n❌ Error with Input/Validation for {video_url}: {ve}")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred for {video_url}: {e}")

# --- BATCH EXECUTION START ---
try:
    urls_to_process = load_video_urls(VIDEO_LIST_PATH)
    print(f"\n🚀 Starting batch analysis for {len(urls_to_process)} videos...")

    for i, url in enumerate(urls_to_process):
        print(f"\n\n--- VIDEO {i+1}/{len(urls_to_process)} ---\n")
        process_single_video(url)
    
    print("\n\n*** BATCH PROCESSING COMPLETE ***")

except Exception as e:
    print(f"\n❌ CRITICAL ERROR in BATCH STARTUP: {e}")